In [14]:
import os
import glob
import math
import random
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import h5py

from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [15]:
DATASET_ROOT = ".\data"   
TRAIN_BUILDINGS = ["building_01", "building_02", "building_03"]
TEST_BUILDING = "building_04"

WINDOW_SIZE = 129
BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-3
STRIDE = 1
PATIENCE = 4
VAL_RATIO = 0.15
NUM_WORKERS = 0
SEED = 42

TARGET_CATEGORIES = ["cold", "laundry", "cooking", "adapter"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
CATEGORY_MAP = {
    "cold": ["fridge", "freezer"],
    "laundry": ["washing_machine", "dryer", "washer_dryer"],
    "cooking": ["dishwasher", "oven", "microwave", "electric_stove", "kettle"],
    "adapter": ["computer", "laptop", "tv", "router", "monitor", "adapter", "charger", "cii-adapter"]
}

DEVICE: cpu


In [16]:
def read_h5_series(file_path, verbose=False):
    with h5py.File(file_path, "r") as f:
        candidates = []

        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                candidates.append((name, obj.shape, obj.dtype))

        f.visititems(visitor)

        if verbose:
            print(f"\n[DEBUG] File: {file_path}")
            for name, shape, dtype in candidates:
                print(f"  dataset={name}, shape={shape}, dtype={dtype}")

        numeric_candidates = []
        for name, shape, dtype in candidates:
            dt = np.dtype(dtype)
            if np.issubdtype(dt, np.number):
                numeric_candidates.append((name, shape, dtype))

        if len(numeric_candidates) == 0:
            raise ValueError(f"No numeric dataset found in {file_path}")

        best_name = None
        best_size = -1

        for name, shape, dtype in numeric_candidates:
            size = int(np.prod(shape)) if len(shape) > 0 else 1
            if size > best_size:
                best_size = size
                best_name = name

        arr = f[best_name][()]

    arr = np.asarray(arr)
    arr = np.squeeze(arr)

    if arr.ndim == 0:
        arr = np.array([arr], dtype=np.float32)

    if arr.ndim > 1:
        if arr.shape[1] > 1:
            arr = arr[:, 0]
        else:
            arr = arr.reshape(-1)

    arr = np.asarray(arr, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    return arr

In [17]:
def list_building_files(building_path):
    files = sorted(glob.glob(os.path.join(building_path, "*.h5")))
    print(f"\nFiles in {building_path}:")
    for f in files:
        print(" -", os.path.basename(f))

In [18]:
def load_building_data(building_path, target_category):
    appliance_files = sorted(glob.glob(os.path.join(building_path, "*.h5")))
    if len(appliance_files) == 0:
        raise FileNotFoundError(f"No .h5 files found in {building_path}")

    series_dict = {}
    for ap_file in appliance_files:
        base = os.path.basename(ap_file).replace(".h5", "").lower()
        arr = read_h5_series(ap_file)
        series_dict[base] = arr

    min_len = min(len(v) for v in series_dict.values())
    for k in series_dict:
        series_dict[k] = series_dict[k][:min_len].astype(np.float32)

    mains = np.zeros(min_len, dtype=np.float32)
    for arr in series_dict.values():
        mains += arr

    target_series = np.zeros(min_len, dtype=np.float32)
    target_names = CATEGORY_MAP[target_category]
    matched = []

    for file_name, arr in series_dict.items():
        if file_name in target_names:
            target_series += arr
            matched.append(file_name)

    print(f"[INFO] Building={os.path.basename(building_path)} | category={target_category} | matched={matched}")

    if len(matched) == 0:
        raise ValueError(
            f"No files matched category '{target_category}' in {building_path}. "
            f"Available files: {list(series_dict.keys())}"
        )

    return mains, target_series

In [19]:
def concat_buildings(buildings, target_category):
    xs = []
    ys = []

    for b in buildings:
        bp = os.path.join(DATASET_ROOT, b)
        x, y = load_building_data(bp, target_category)
        xs.append(x)
        ys.append(y)

    return np.concatenate(xs), np.concatenate(ys)

In [20]:
def relative_mae_percent(y_true, y_pred):
    denom = np.mean(np.abs(y_true)) + 1e-8
    return 100.0 * np.mean(np.abs(y_true - y_pred)) / denom

def make_sample_weights(y, active_threshold):
    y_flat = y.reshape(-1)
    active = (y_flat > active_threshold).astype(np.float32)
    w = 1.0 + 2.0 * active
    return w.reshape(-1, 1).astype(np.float32)

In [21]:
class WindowedSeq2PointDataset(Dataset):
    def __init__(self, x, y, indices, window_size, x_mean, x_std, y_mean, y_std, active_threshold=None):
        self.x = x.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.window_size = window_size
        self.half = window_size // 2
        self.x_mean = float(x_mean)
        self.x_std = float(x_std) + 1e-8
        self.y_mean = float(y_mean)
        self.y_std = float(y_std) + 1e-8
        self.active_threshold = active_threshold

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        center = self.indices[idx]
        xw = self.x[center - self.half : center + self.half + 1]
        yt = self.y[center]

        xw = (xw - self.x_mean) / self.x_std
        yt_scaled = (yt - self.y_mean) / self.y_std

        if self.active_threshold is None:
            return torch.tensor(xw, dtype=torch.float32), torch.tensor([yt_scaled], dtype=torch.float32)

        weight = 3.0 if yt > self.active_threshold else 1.0
        return (
            torch.tensor(xw, dtype=torch.float32),
            torch.tensor([yt_scaled], dtype=torch.float32),
            torch.tensor([weight], dtype=torch.float32)
        )

In [22]:
class SmallSeq2PointRegressor(nn.Module):
    def __init__(self, window_size=129):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7),
            nn.ReLU(),
            nn.Conv1d(16, 24, kernel_size=5),
            nn.ReLU(),
            nn.Conv1d(24, 32, kernel_size=5),
            nn.ReLU()
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, window_size)
            out = self.features(dummy)
            flat_dim = out.view(1, -1).shape[1]

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.features(x)
        return self.regressor(x)

In [23]:
class WeightedHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta

    def forward(self, pred, target, weight=None):
        err = pred - target
        abs_err = torch.abs(err)
        quadratic = torch.minimum(abs_err, torch.tensor(self.delta, device=pred.device))
        linear = abs_err - quadratic
        loss = 0.5 * quadratic**2 + self.delta * linear

        if weight is not None:
            loss = loss * weight

        return loss.mean()

In [24]:
def evaluate_regression(model, loader, y_mean, y_std, device):
    model.eval()
    preds = []
    trues = []

    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                xb, yb, _ = batch
            else:
                xb, yb = batch

            xb = xb.to(device, non_blocking=True)
            pred = model(xb).cpu().numpy().reshape(-1)
            true = yb.cpu().numpy().reshape(-1)

            preds.append(pred)
            trues.append(true)

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)

    preds_inv = preds * y_std + y_mean
    trues_inv = trues * y_std + y_mean

    mae = mean_absolute_error(trues_inv, preds_inv)
    r2 = r2_score(trues_inv, preds_inv)
    rel_mae = 100.0 * np.mean(np.abs(trues_inv - preds_inv)) / (np.mean(np.abs(trues_inv)) + 1e-8)

    return mae, r2, rel_mae

In [25]:
def train_one_category(target_category):
    print("\n" + "="*70)
    print("Training category:", target_category)
    print("="*70)

    x_train_all, y_train_all = concat_buildings(TRAIN_BUILDINGS, target_category)
    x_test, y_test = concat_buildings([TEST_BUILDING], target_category)

    half = WINDOW_SIZE // 2
    train_stride = 120
    eval_stride = 60
    max_train_samples = 50000
    max_val_samples = 10000
    max_test_samples = 15000

    train_indices_all = np.arange(half, len(x_train_all) - half, train_stride)
    test_indices = np.arange(half, len(x_test) - half, eval_stride)

    train_idx, val_idx = train_test_split(
        train_indices_all, test_size=VAL_RATIO, random_state=SEED
    )

    rng = np.random.default_rng(SEED)

    if len(train_idx) > max_train_samples:
        train_idx = np.sort(rng.choice(train_idx, size=max_train_samples, replace=False))

    if len(val_idx) > max_val_samples:
        val_idx = np.sort(rng.choice(val_idx, size=max_val_samples, replace=False))

    if len(test_indices) > max_test_samples:
        test_indices = np.sort(rng.choice(test_indices, size=max_test_samples, replace=False))

    x_mean = x_train_all.mean()
    x_std = x_train_all.std() + 1e-8
    y_mean = y_train_all[train_idx].mean()
    y_std = y_train_all[train_idx].std() + 1e-8

    positive_y = y_train_all[train_idx][y_train_all[train_idx] > 0]
    active_threshold = max(1.0, 0.05 * positive_y.mean()) if len(positive_y) > 0 else 1.0

    train_ds = WindowedSeq2PointDataset(
        x_train_all, y_train_all, train_idx, WINDOW_SIZE,
        x_mean, x_std, y_mean, y_std, active_threshold=active_threshold
    )
    val_ds = WindowedSeq2PointDataset(
        x_train_all, y_train_all, val_idx, WINDOW_SIZE,
        x_mean, x_std, y_mean, y_std, active_threshold=None
    )
    test_ds = WindowedSeq2PointDataset(
        x_test, y_test, test_indices, WINDOW_SIZE,
        x_mean, x_std, y_mean, y_std, active_threshold=None
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = SmallSeq2PointRegressor(window_size=WINDOW_SIZE).to(DEVICE)
    criterion = WeightedHuberLoss(delta=1.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_val_mae = float("inf")
    best_epoch = 0
    best_state = None
    wait = 0

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for xb, yb, wb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            wb = wb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb, wb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)

        train_loss = total_loss / len(train_loader.dataset)
        val_mae, val_r2, val_rel = evaluate_regression(model, val_loader, y_mean, y_std, DEVICE)

        print(
            f"Epoch {epoch+1:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_MAE={val_mae:.4f} | "
            f"val_R2={val_r2:.4f}"
        )

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    test_mae, test_r2, test_rel = evaluate_regression(model, test_loader, y_mean, y_std, DEVICE)

    print(f"Best epoch: {best_epoch}")
    print(f"Final Test MAE: {test_mae:.4f}")
    print(f"Final Test R2 : {test_r2:.4f}")
    print(f"Relative MAE %: {test_rel:.2f}")

    return {
        "category": target_category,
        "test_building": TEST_BUILDING,
        "mae": test_mae,
        "r2": test_r2,
        "relative_mae_pct": test_rel,
        "true_mean_power": float(y_test.mean()),
        "best_epoch": best_epoch,
        "status": "ok"
    }

In [26]:
all_results = []

for cat in TARGET_CATEGORIES:
    try:
        res = train_one_category(cat)
        all_results.append(res)
    except Exception as e:
        print(f"Category {cat} failed: {e}")
        all_results.append({
            "category": cat,
            "test_building": TEST_BUILDING,
            "mae": np.nan,
            "r2": np.nan,
            "relative_mae_pct": np.nan,
            "true_mean_power": np.nan,
            "best_epoch": -1,
            "status": f"failed: {str(e)}"
        })

results_df = pd.DataFrame(all_results).sort_values(by="r2", ascending=False)
print("\nFinal results table:")
print(results_df)


Training category: cold
[INFO] Building=building_01 | category=cold | matched=['freezer']
[INFO] Building=building_02 | category=cold | matched=['fridge']
[INFO] Building=building_03 | category=cold | matched=['fridge']
[INFO] Building=building_04 | category=cold | matched=['fridge']
Epoch 01/20 | train_loss=0.2143 | val_MAE=10.5145 | val_R2=0.7897
Epoch 02/20 | train_loss=0.1076 | val_MAE=8.7047 | val_R2=0.8428
Epoch 03/20 | train_loss=0.0956 | val_MAE=8.3928 | val_R2=0.8447
Epoch 04/20 | train_loss=0.0855 | val_MAE=6.9576 | val_R2=0.8412
Epoch 05/20 | train_loss=0.0803 | val_MAE=7.1840 | val_R2=0.8724
Epoch 06/20 | train_loss=0.0777 | val_MAE=6.7692 | val_R2=0.8606
Epoch 07/20 | train_loss=0.0740 | val_MAE=7.6682 | val_R2=0.8689
Epoch 08/20 | train_loss=0.0740 | val_MAE=7.1677 | val_R2=0.8693
Epoch 09/20 | train_loss=0.0708 | val_MAE=7.1840 | val_R2=0.8577
Epoch 10/20 | train_loss=0.0674 | val_MAE=6.6154 | val_R2=0.8682
Epoch 11/20 | train_loss=0.0661 | val_MAE=6.7762 | val_R2=0.877